### Install Libraries

In [16]:
pip install xgboost scikit-learn pandas numpy joblib

Note: you may need to restart the kernel to use updated packages.


### Import Libraries

In [17]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

from xgboost import XGBRegressor

### Load Dataset

In [18]:
ev_df=pd.read_csv(r"C:\Users\shaik\Downloads\resale_value.csv")
ev_df

,brand,vehicle_type,battery_health_pct,range_km,top_speed_kmph,odometer_km,warranty_remaining_years,annual_maintenance_cost,Purchase_Year,Current_Year,Purchase_Price_L,Resale_Value_L,Initial_Mileage,Current_Mileage,value_for_money_score
0,BYD,Car,81,105,144,140236,1,32026,2023,2025,2093043,1475081,202,166,0.4
1,Ola,Bike,61,343,160,36713,5,29149,2018,2025,90099,34748,402,348,0.4
2,TVS,Bike,63,478,132,137457,3,27366,2024,2025,109294,90486,226,214,0.7
3,BYD,Bike,72,440,95,47977,1,23723,2024,2025,144702,120813,198,182,0.7
4,Hyundai,Car,91,258,175,27678,5,33318,2020,2025,2078297,1016731,348,249,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,Tata,Bike,77,127,102,79607,0,34758,2023,2025,180454,128494,279,270,0.4
19996,TVS,Bike,88,447,145,116383,3,5677,2024,2025,158370,128863,184,157,0.9
19997,TVS,Car,86,436,100,148117,0,27052,2018,2025,2407574,828233,375,328,0.5
19998,Kia,Bike,94,385,135,79820,5,26084,2018,2025,136591,36612,323,296,0.6


### Feature Engineering

#### Adding new columns

In [19]:
ev_df["vehicle_age"] = ev_df["Current_Year"] - ev_df["Purchase_Year"]

ev_df["usage_per_year"] = ev_df["odometer_km"] / (ev_df["vehicle_age"] + 1)

ev_df["mileage_drop"] = ev_df["Initial_Mileage"] - ev_df["Current_Mileage"]

In [20]:
ev_df.head()

,brand,vehicle_type,battery_health_pct,range_km,top_speed_kmph,odometer_km,warranty_remaining_years,annual_maintenance_cost,Purchase_Year,Current_Year,Purchase_Price_L,Resale_Value_L,Initial_Mileage,Current_Mileage,value_for_money_score,vehicle_age,usage_per_year,mileage_drop
0,BYD,Car,81,105,144,140236,1,32026,2023,2025,2093043,1475081,202,166,0.4,2,46745.333333,36
1,Ola,Bike,61,343,160,36713,5,29149,2018,2025,90099,34748,402,348,0.4,7,4589.125000,54
2,TVS,Bike,63,478,132,137457,3,27366,2024,2025,109294,90486,226,214,0.7,1,68728.500000,12
3,BYD,Bike,72,440,95,47977,1,23723,2024,2025,144702,120813,198,182,0.7,1,23988.500000,16
4,Hyundai,Car,91,258,175,27678,5,33318,2020,2025,2078297,1016731,348,249,0.5,5,4613.000000,99


#### Define Base Features

In [21]:
features = [
    "brand",
    "vehicle_type",
    "battery_health_pct",
    "range_km",
    "top_speed_kmph",
    "odometer_km",
    "warranty_remaining_years",
    "annual_maintenance_cost",
    "Purchase_Price_L",
    "vehicle_age",
    "usage_per_year",
    "mileage_drop"
]

X = ev_df[features]

y_condition = ev_df["value_for_money_score"]
y_price = ev_df["Resale_Value_L"]

#### Preprocessing

In [22]:
categorical_cols = ["brand", "vehicle_type"]
numeric_cols = [col for col in features if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

### Train-Test Split ( Spliting for Both Targets)

In [23]:
X_train, X_test, y_cond_train, y_cond_test, y_price_train, y_price_test = train_test_split(
    X, y_condition, y_price,
    test_size=0.2,
    random_state=42
)

### 1.Vehicle Condition Model

In [24]:
xgb_condition = XGBRegressor(
    n_estimators=120,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.5,
    colsample_bytree=0.5,
    reg_alpha=4,
    reg_lambda=6,
    min_child_weight=7,
    random_state=42
)

condition_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_condition)
])

condition_pipeline.fit(X_train, y_cond_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['brand', 'vehicle_type']),
                                                 ('num', 'passthrough',
                                                  ['battery_health_pct',
                                                   'range_km', 'top_speed_kmph',
                                                   'odometer_km',
                                                   'warranty_remaining_years',
                                                   'annual_maintenance_cost',
                                                   'Purchase_Price_L',
                                                   'vehicle_age',
                                                   'usage_per_year',
                                                   'mileage_drop'])])),
                ('mode...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=2, max_leaves=None, min_child_weight=7,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=120,
                              n_jobs=None, num_parallel_tree=None, ...))])

### Evaluate Vehicle Condition Model

In [25]:
pred_cond_test = condition_pipeline.predict(X_test)

print("Condition Model R2:", r2_score(y_cond_test, pred_cond_test))
print("Condition Model MAE:", mean_absolute_error(y_cond_test, pred_cond_test))

Condition Model R2: 0.8759650588534582
Condition Model MAE: 0.04178264651596545


### Proper Stacking (NO FULL DATA PREDICTION)

In [26]:
pred_cond_train = condition_pipeline.predict(X_train)
pred_cond_test = condition_pipeline.predict(X_test)

X_train_price = X_train.copy()
X_test_price = X_test.copy()

X_train_price["predicted_condition"] = pred_cond_train
X_test_price["predicted_condition"] = pred_cond_test

### 2.Price Model

In [27]:
xgb_price = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=1,
    subsample=0.3,
    colsample_bytree=0.3,
    reg_alpha=6,
    reg_lambda=8,
    min_child_weight=8,
    random_state=42
)

price_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_price)
])

price_pipeline.fit(X_train_price, y_price_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['brand', 'vehicle_type']),
                                                 ('num', 'passthrough',
                                                  ['battery_health_pct',
                                                   'range_km', 'top_speed_kmph',
                                                   'odometer_km',
                                                   'warranty_remaining_years',
                                                   'annual_maintenance_cost',
                                                   'Purchase_Price_L',
                                                   'vehicle_age',
                                                   'usage_per_year',
                                                   'mileage_drop'])])),
                ('mode...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=1, max_leaves=None, min_child_weight=8,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=100,
                              n_jobs=None, num_parallel_tree=None, ...))])

### Evaluate Price Model

In [28]:
pred_price = price_pipeline.predict(X_test_price)

print("Price Model R2:", r2_score(y_price_test, pred_price))
print("Price Model MAE:", mean_absolute_error(y_price_test, pred_price))

Price Model R2: 0.852519690990448
Price Model MAE: 177381.09375


### Save Models

In [29]:
joblib.dump(condition_pipeline, "condition_model.pkl")
joblib.dump(price_pipeline, "price_model.pkl")

['price_model.pkl']

### Final Prediction + Advice Function

In [54]:
def predict_and_advise(user_data, user_price):

    user_df = pd.DataFrame([user_data])

    user_df["vehicle_age"] = user_df["Current_Year"] - user_df["Purchase_Year"]
    user_df["usage_per_year"] = user_df["odometer_km"] / (user_df["vehicle_age"] + 1)
    user_df["mileage_drop"] = user_df["Initial_Mileage"] - user_df["Current_Mileage"]

    user_X = user_df[features]

    # Condition Prediction
    predicted_condition = condition_pipeline.predict(user_X)[0]
    condition_percentage = predicted_condition * 100

    user_df["predicted_condition"] = predicted_condition
    user_X_price = user_df[features + ["predicted_condition"]]

    # Price Prediction
    predicted_price = price_pipeline.predict(user_X_price)[0]

    # Business Rule
    if predicted_price > user_data["Purchase_Price_L"]:
        predicted_price = user_data["Purchase_Price_L"] * 0.9

    # Comparison
    difference = user_price - predicted_price
    percentage_diff = (difference / predicted_price) * 100

    # Advice Logic
    if percentage_diff > 10:
        advice = "Overpriced"
    elif percentage_diff < -10:
        advice = "Underpriced (Good Deal)"
    else:
        advice = "Fair Price"

    # 🔥 INSIGHT SECTION ADD HERE
    if advice == "Overpriced":
        insight = "⚠️ The vehicle appears overpriced mainly due to pricing exceeding expected market value."
    elif advice == "Underpriced (Good Deal)":
        insight = "✅ The vehicle seems undervalued compared to predicted resale price, indicating a potential good deal."
    else:
        insight = "👍 The vehicle price aligns closely with predicted market value."

    # OUTPUT
    print("\n=========================================")
    print("💰 Prediction Result")
    print("=========================================")
    print("Vehicle Condition Score:", round(condition_percentage, 2), "%")
    print("Predicted Resale Price:", round(predicted_price, 2))
    print("User Price:", user_price)
    print("Price Difference (%):", round(percentage_diff, 2))
    print("Recommendation:", advice)
    print("\n📌 Insight:")
    print(insight)
    print("=========================================")

In [55]:
def run_ev_resale_prediction():

    print("\n🚗 EV Resale Price Prediction System")
    print("=========================================")

    brand = input("Enter Brand:  ")
    vehicle_type = input("Enter Vehicle Type (Car/Bike/SUV):  ")
    battery_health = float(input("Enter Battery Health (%):  "))
    range_km = float(input("Enter Range (km):  "))
    top_speed = float(input("Enter Top Speed (kmph):  "))
    odometer = float(input("Enter Odometer Reading (km):  "))
    warranty = float(input("Enter Warranty Remaining (years):  "))
    maintenance = float(input("Enter Annual Maintenance Cost:  "))
    purchase_price = float(input("Enter Purchase Price (Lakhs):  "))
    purchase_year = int(input("Enter Purchase Year:  "))
    current_year = int(input("Enter Current Year:  "))
    initial_mileage = float(input("Enter Initial Mileage (km):  "))
    current_mileage = float(input("Enter Current Mileage (km):  "))
    user_price = float(input("Enter Expected Resale Price:  "))

    user_input = {
        "brand": brand,
        "vehicle_type": vehicle_type,
        "battery_health_pct": battery_health,
        "range_km": range_km,
        "top_speed_kmph": top_speed,
        "odometer_km": odometer,
        "warranty_remaining_years": warranty,
        "annual_maintenance_cost": maintenance,
        "Purchase_Price_L": purchase_price,
        "Purchase_Year": purchase_year,
        "Current_Year": current_year,
        "Initial_Mileage": initial_mileage,
        "Current_Mileage": current_mileage
    }

    # print("\n=========================================")
    # print("💰 Prediction Result")
    # print("=========================================")

    predict_and_advise(user_input, user_price)

In [56]:
run_ev_resale_prediction()


🚗 EV Resale Price Prediction System


Enter Brand:   BYD
Enter Vehicle Type (Car/Bike/SUV):   Car
Enter Battery Health (%):   81
Enter Range (km):   105
Enter Top Speed (kmph):   144
Enter Odometer Reading (km):   140236
Enter Warranty Remaining (years):   1
Enter Annual Maintenance Cost:   32026
Enter Purchase Price (Lakhs):   2093043
Enter Purchase Year:   2023
Enter Current Year:   2025
Enter Initial Mileage (km):   202
Enter Current Mileage (km):   166
Enter Expected Resale Price:   1475081



💰 Prediction Result
Vehicle Condition Score: 44.91 %
Predicted Resale Price: 1228616.6
User Price: 1475081.0
Price Difference (%): 20.06
Recommendation: Overpriced

📌 Insight:
⚠️ The vehicle appears overpriced mainly due to pricing exceeding expected market value.
